**Подготовка датасета на основе файла final_report_2026-01-27.csv**

In [1]:
%cd ..
%pwd

D:\Python_2\MLogic_project\Yuperio\prognoz_prodaj_12_2025


'D:\\Python_2\\MLogic_project\\Yuperio\\prognoz_prodaj_12_2025'

In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb

from utils import df_date_convert, create_dataset, evaluate_model
from sales_dataset import SalesDataset
from window_features import FeatureExtractor

In [3]:
df_csv = pd.read_csv('./etl/final_report_2026-01-27.csv')
df_csv = df_csv[(df_csv['exit_type']=='Продажа') & (df_csv['Territory'].str.contains('moscow', case=False, na=False)) ]
df_csv = df_csv[df_csv['12_2025'].notna()]

df_conv = df_date_convert(df_csv)
df_conv

,location_mdlp_id,Адрес ЛПУ,Territory,exit_type,2023-01-01,2023-02-01,2023-03-01,2023-04-01,2023-05-01,2023-06-01,...,2025-03-01,2025-04-01,2025-05-01,2025-06-01,2025-07-01,2025-08-01,2025-09-01,2025-10-01,2025-11-01,2025-12-01
1,166,"Москва г, Митинский 3-й пер., 4",MR PC BU CVM Moscow_3,Продажа,7.0,8.0,11.0,15.0,10.0,9.0,...,16.0,30.0,19.0,27.0,20.0,24.0,20.0,33.0,32.0,34.0
3,167,"Серпухов г, Московское ш., 64",MR PC BU CVM Moscow_11,Продажа,1.0,1.0,2.0,2.0,2.0,NaN,...,5.0,2.0,6.0,NaN,7.0,2.0,2.0,6.0,2.0,6.0
5,169,"Москва г, Маршала Федоренко ул., 6",MR PC BU CVM Moscow_SAO,Продажа,2.0,1.0,4.0,6.0,3.0,1.0,...,34.0,38.0,21.0,27.0,18.0,26.0,25.0,33.0,20.0,25.0
7,170,"Москва г, Скульптора Мухиной ул., 14",MR PC BU CVM Moscow_2,Продажа,2.0,5.0,2.0,7.0,8.0,7.0,...,13.0,26.0,10.0,14.0,18.0,11.0,11.0,16.0,15.0,14.0
9,171,"Москва г, Героев Панфиловцев ул., 8",MR PC BU CVM Moscow_3,Продажа,5.0,4.0,1.0,3.0,4.0,NaN,...,NaN,NaN,3.0,2.0,NaN,NaN,NaN,2.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103144,535646,"Юность п, нет, 2",MR PC BU CVM Moscow_UAO_2,Продажа,NaN,NaN,NaN,NaN,NaN,NaN,...,2.0,NaN,NaN,3.0,3.0,2.0,3.0,2.0,2.0,8.0
103201,535785,"Москва г, Вольская 1-я ул., 3",MR PC BU CVM Moscow_UVAO_1,Продажа,NaN,NaN,NaN,NaN,NaN,NaN,...,2.0,2.0,2.0,NaN,2.0,NaN,NaN,NaN,1.0,3.0
103203,535786,"Одинцово г, Красногорское ш., 15",MR PC BU CVM Moscow_SZAO_1,Продажа,NaN,NaN,NaN,NaN,NaN,NaN,...,2.0,4.0,2.0,2.0,1.0,3.0,4.0,2.0,NaN,2.0
103221,535826,"Москва г, Новаторов ул., 5",MR PC BU CVM Moscow_5,Продажа,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,6.0,4.0,3.0,1.0,5.0,1.0,2.0,4.0,6.0


In [4]:
sale_ds = SalesDataset(df_conv)
# sale_ds.all_windows_to_predict

**Обучение модели (регрессия)**

In [11]:
X, y = sale_ds.data_regress
dataset = create_dataset(X,y)

# Параметры модели
params = {
    'objective': 'regression',
    'metric':  ['rmse', 'mae'],
    'learning_rate': 0.01,
    'max_depth': 5,
    'num_leaves': 31,
    'verbosity': -1
}

# Обучение с ранней остановкой
model_early_stop = lgb.train(
    params,
    dataset.train_data,
    valid_sets=[dataset.valid_data],
    num_boost_round=1000,
    callbacks=[
        lgb.early_stopping(stopping_rounds=300), # Если в течение 300 итераций подряд метрика не стала лучше
        lgb.log_evaluation(50)  # Выводим логи каждые 50 итераций
    ]
)

model_early_stop.save_model("lgb_model_regress_2.txt")

Training until validation scores don't improve for 3000 rounds
[50]	valid_0's rmse: 8.34414	valid_0's l1: 6.01047
[100]	valid_0's rmse: 7.11994	valid_0's l1: 5.10083
[150]	valid_0's rmse: 6.62709	valid_0's l1: 4.70914
[200]	valid_0's rmse: 6.45435	valid_0's l1: 4.54811
[250]	valid_0's rmse: 6.39124	valid_0's l1: 4.48262
[300]	valid_0's rmse: 6.3754	valid_0's l1: 4.45529
[350]	valid_0's rmse: 6.37247	valid_0's l1: 4.44278
[400]	valid_0's rmse: 6.37819	valid_0's l1: 4.43733
[450]	valid_0's rmse: 6.38804	valid_0's l1: 4.4332
[500]	valid_0's rmse: 6.39601	valid_0's l1: 4.43021
[550]	valid_0's rmse: 6.40445	valid_0's l1: 4.42891
[600]	valid_0's rmse: 6.41502	valid_0's l1: 4.42865
[650]	valid_0's rmse: 6.42707	valid_0's l1: 4.42912
[700]	valid_0's rmse: 6.44217	valid_0's l1: 4.43036
[750]	valid_0's rmse: 6.45853	valid_0's l1: 4.43169
[800]	valid_0's rmse: 6.47496	valid_0's l1: 4.43299
[850]	valid_0's rmse: 6.49154	valid_0's l1: 4.43439
[900]	valid_0's rmse: 6.50846	valid_0's l1: 4.43594
[950

In [12]:
# Тестирование модели
# 1. Загрузка модели
loaded_model = lgb.Booster(model_file="lgb_model_regress_2.txt")

# 2. Предсказание на тестовых данных
y_pred = loaded_model.predict(dataset.X_test)
early_results = evaluate_model(dataset.y_test, y_pred, "LightGBM с ранней остановкой")

print(f"\nКоличество использованных деревьев: {model_early_stop.best_iteration}")



LightGBM с ранней остановкой Результаты:
Среднеквадратичная ошибка (MSE): 40.6069
Корень из MSE (RMSE): 6.3724
Средняя абсолютная ошибка (MAE): 4.4449
Средняя абсолютная процентная ошибка (MAPE): 49.50%
Коэффициент детерминации (R²): 0.6661

Количество использованных деревьев: 338


**Прогнозироване продаж**

In [13]:
# Получение окна продаж
res = sale_ds.get_last_window(mdlp_id=127357)
print(f'{res.month_predict=}, {res.window_sale=}\n')

# Получение фичей
ft = FeatureExtractor()
features = ft.compute_window(res.window_sale, num_month=res.month_predict)
print(features)

# Прогноз продаж
X = np.array(features.to_list(), dtype=float)
X = X.reshape(1,-1)
#
model_file = 'lgb_model_regress_2.txt'
model = lgb.Booster(model_file=model_file)
y_pred = model.predict(X)
print(f'\n{y_pred=}')

res.month_predict=1, res.window_sale=[7.0, 2.0, 3.0, 8.0, 3.0, 15.0, 5.0, 2.0, 11.0, 22.0, 7.0, 9.0]

Features(MeanW=7.833333333333333, StdW=5.683797634993311, CV=0.7255910948173041, RelLast=1.148936023540082, ZLast=0.20526178733424466, Delta1=2.0, LogRet1=0.2513143965348783, Mom3=-2.0, Mom6=-6.0, Slope10=0.7062937062937065, R2_Trend=0.1840128916655143, ZResLast=-0.5293719408218017, UpShare=0.5454545454545454, SignChanges=7.0, MaxDrawdown=0.8666666088888928, MonthCos=0.8660254037844387, MonthSin=0.49999999999999994, Lag12Target=7.0)

y_pred=array([10.34194999])
